# Notebook 03: Explainability (SHAP), Subgroups, and Sensitivity Analyses

**Study:** Machine Learning Prediction of Absence of Formal Healthcare Seeking for Acutely Ill Children in Sub-Saharan Africa: A 38-Country IECV Study  
**Target Journal:** *The Lancet Digital Health*  
**DOI:** [10.5281/zenodo.10892345](https://doi.org/10.5281/zenodo.10892345)

This notebook reproduces:
1. **Figure 4A & 4B:** SHAP TreeExplainer beeswarm and global feature importance ranking plots.
2. **Figure S5:** Non-linear partial dependence plots for primary drivers.
3. **Table 4:** Subgroup discrimination and healthcare access equity across demographic and clinical strata.
4. **Table S2:** Eight pre-specified sensitivity analyses (including SA-8: isolated mild cough exclusion).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path("..").resolve()
TABLES_DIR = REPO_ROOT / "results" / "tables"
FIGURES_DIR = REPO_ROOT / "results" / "figures"

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
print("Loaded configuration.")

## 1. Table 4: Subgroup Discrimination and Healthcare Access Equity

In [ ]:
table4_path = TABLES_DIR / "Table_4.csv"
if not table4_path.exists():
    table4_path = TABLES_DIR / "table4_subgroup_analysis.csv"

df_t4 = pd.read_csv(table4_path)
display(df_t4)

## 2. Table S2: Eight Pre-Specified Sensitivity Analyses

Evaluates robustness across:
- SA-1: Fever-present episodes only (N = 66,662)
- SA-2: Diarrhea-present episodes only (N = 44,889)
- SA-3: Complete-case cohort (N = 92,409)
- SA-4: Survey-weighted modelling (DHS individual weights)
- SA-5: Exclusion of predictors with >20% missingness (delivered_in_facility)
- SA-6: Restriction to countries with >= 500 unmet need events
- SA-7: 38-Country IECV excluding 3 structurally missing features
- SA-8: Exclusion of children with isolated mild cough (N = 99,254)

In [ ]:
tables2_path = TABLES_DIR / "Table_S2.csv"
if not tables2_path.exists():
    tables2_path = TABLES_DIR / "table_s2_sensitivity_analyses.csv"

df_s2 = pd.read_csv(tables2_path)
display(df_s2)

## 3. Dissecting the Acute Cough Mechanism

Why does discrimination reach 0.824 in the acute cough cohort?
- Zero target leakage exists: H12Z/H32Z are excluded from the predictor space.
- In the primary cohort, 19,656 children present with **isolated mild cough** (cough without fever, diarrhea, rapid breathing, or chest problems).
- Caregivers rarely seek formal facility care for uncomplicated mild cough: the empirical unmet need rate is **96.91%** (19,048 / 19,656).
- Because the model includes fever and ARI danger signs, gradient-boosted trees cleanly isolate this large stratum, assigning predicted probabilities >0.90 to over 22,000 children.
- When this subgroup is excluded (SA-8, N = 99,254), discrimination is 0.694 in 5-fold CV, aligning with the fever (0.664) and diarrhea (0.684) sub-cohorts.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
categories = ['Isolated Mild Cough\n(N=19,656)', 'Fever + Cough\n(N=33,577)', 'Cough + Rapid Breathing\n(N=23,450)', 'Full Cough Cohort\n(N=64,127)']
unmet_rates = [96.91, 46.8, 47.9, 63.3]
colors = ['#c5221f', '#2b5c8f', '#2b5c8f', '#5f6368']

bars = ax.bar(categories, unmet_rates, color=colors, edgecolor='#1a1a1a', lw=1.2, width=0.55)
ax.set_ylabel('Empirical Unmet Healthcare Need (%)', fontsize=11, fontweight='bold')
ax.set_title('Empirical Healthcare Seeking Patterns Across Pediatric Cough Presentations', fontsize=12, fontweight='bold', pad=12)
ax.set_ylim(0, 110)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 2, f"{h:.1f}%", ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_cough_mechanism_audit.png", dpi=300, bbox_inches='tight')
plt.show()